# Document Ingestion & Parsing — Hands-On

Offline lab: reconstruct reading order, tables, chunk metadata, and a toy quality score from simulated layout boxes.

## 0. Setup

In [ ]:
%pip install -q numpy\nimport numpy as np\nfrom collections import defaultdict

## 1. Reading-order reconstruction from layout boxes

In [ ]:
def reading_order(boxes, y_tol=8):
    rows=[]
    for b in sorted(boxes, key=lambda x:(x['y0'], x['x0'])):
        for row in rows:
            if abs(row[0]['y0'] - b['y0']) <= y_tol:
                row.append(b); break
        else:
            rows.append([b])
    out=[]
    for row in rows: out.extend(sorted(row, key=lambda x:x['x0']))
    return out
boxes=[{'text':'Total','x0':320,'y0':200},{'text':'Invoice','x0':40,'y0':40},{'text':'Acme','x0':40,'y0':120},{'text':'$42','x0':410,'y0':200}]
ordered=reading_order(boxes)
print(' | '.join(b['text'] for b in ordered))
assert ordered[0]['text']=='Invoice'

## 2. Table reconstruction

In [ ]:
def reconstruct_table(cells):
    grid=[["" for _ in range(max(c['col'] for c in cells)+1)] for _ in range(max(c['row'] for c in cells)+1)]
    for c in cells: grid[c['row']][c['col']] = c['text'].strip()
    return grid
def table_to_markdown(grid):
    return "\n".join(["| " + " | ".join(grid[0]) + " |", "| " + " | ".join(["---"]*len(grid[0])) + " |"] + ["| " + " | ".join(r) + " |" for r in grid[1:]])
cells=[{'row':0,'col':0,'text':'Item'},{'row':0,'col':1,'text':'Price'},{'row':1,'col':0,'text':'Widget'},{'row':1,'col':1,'text':'$10'}]
print(table_to_markdown(reconstruct_table(cells)))
assert reconstruct_table(cells)[1][1] == '$10'

## 3. Chunking-aware metadata

In [ ]:
def make_chunks(blocks, max_chars=60):
    chunks=[]; cur=''; meta=[]
    for b in blocks:
        if len(cur)+len(b['text']) > max_chars and cur:
            chunks.append({'text':cur.strip(), 'meta':meta}); cur=''; meta=[]
        cur += b['text'] + ' '; meta.append((b['page'], b['type']))
    if cur: chunks.append({'text':cur.strip(), 'meta':meta})
    return chunks
blocks=[{'text':'Invoice Acme','page':1,'type':'title'}, {'text':'Item Widget Price ten','page':1,'type':'table'}]
print(make_chunks(blocks))

## 4. Parse quality score

In [ ]:
def quality(blocks):
    empty=sum(not b['text'].strip() for b in blocks)
    tables=sum(b.get('type')=='table' for b in blocks)
    has_page=sum('page' in b for b in blocks)
    return round(1 - 0.2*empty + 0.05*tables + 0.05*has_page/len(blocks), 2)
print('quality', quality(blocks))
assert quality(blocks) > 1.0

## 5. Exercises and links
1. Add OCR confidence and drop low-confidence spans.
2. Preserve section headings in each chunk.
3. Compare table-as-markdown vs table-as-JSON.

Cross-link: [[02 Literature Notes/LLM Engineering/Chunking Strategies]].